In [0]:
%run /Workspace/Users/patelrahul2614@gmail.com/IAP-MainExam/TPCH-Sales&SupplyChain/00_setup

In [0]:
catalog = dbutils.widgets.get("catalog")

In [0]:
from pyspark.sql.functions import col, when

table_name = "region"
checkpoint_path = f"/Volumes/{catalog}/silver/checkpoints/{table_name}"
silver_table = f"{catalog}.silver.{table_name}"
quarantine_table = f"{catalog}.silver.{table_name}_quarantine"

# 1. Create silver table if not exists (original columns only, no meta)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {silver_table}
    USING DELTA
    AS SELECT r_regionkey, r_name, r_comment
    FROM {catalog}.bronze.{table_name} WHERE 1=0
""")

# 2. Create quarantine table if not exists (original columns + _rejection_reason)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {quarantine_table}
    USING DELTA
    AS SELECT r_regionkey, r_name, r_comment,
              CAST(NULL AS STRING) AS _rejection_reason
    FROM {catalog}.bronze.{table_name} WHERE 1=0
""")

# 3. Read stream from bronze with CDF
source_df = (spark.readStream
    .format("delta")
    .option("readChangeFeed", "true")
    .table(f"{catalog}.bronze.{table_name}"))

# Columns to keep (exclude CDF + meta columns)
exclude_cols = {"_change_type", "_commit_version", "_commit_timestamp",
                "_data_source", "_ingested_at", "_job_run_id"}
silver_cols = [c for c in source_df.columns if c not in exclude_cols]

# 4. Apply validation rules
#    Rule 1: Primary key (r_regionkey) IS NOT NULL
#    Rule 2: Name (r_name) IS NOT NULL AND not empty
#    Only process inserts and update_postimages (not deletes or preimages)

valid_df = source_df.filter(
    col("_change_type").isin("insert", "update_postimage") &
    col("r_regionkey").isNotNull() &
    col("r_name").isNotNull() &
    (col("r_name") != "")
).select(*silver_cols)

invalid_df = source_df.filter(
    col("_change_type").isin("insert", "update_postimage") &
    (
        col("r_regionkey").isNull() |
        col("r_name").isNull() |
        (col("r_name") == "")
    )
).select(*silver_cols).withColumn("_rejection_reason",
    when(col("r_regionkey").isNull(), "Primary key (r_regionkey) is NULL")
    .when(col("r_name").isNull(), "Name (r_name) is NULL")
    .otherwise("Name (r_name) is empty")
)

# 5. Write valid rows to silver via MERGE (SCD Type 1)
def merge_region(batch_df, batch_id):
    if batch_df.count() > 0:
        batch_df.createOrReplaceTempView("region_updates")
        spark.sql(f"""
            MERGE INTO {silver_table} t
            USING region_updates s
            ON t.r_regionkey = s.r_regionkey
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)

valid_stream = (valid_df.writeStream
    .foreachBatch(merge_region)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start())

# 6. Write invalid rows to quarantine
quarantine_checkpoint = f"{checkpoint_path}_quarantine"
dbutils.fs.mkdirs(quarantine_checkpoint)

def quarantine_region(batch_df, batch_id):
    if batch_df.count() > 0:
        batch_df.write.format("delta").mode("append").saveAsTable(quarantine_table)

invalid_stream = (invalid_df.writeStream
    .foreachBatch(quarantine_region)
    .option("checkpointLocation", quarantine_checkpoint)
    .trigger(availableNow=True)
    .start())

# 7. Wait for both streams to complete
valid_stream.awaitTermination()
print(f"  - {table_name}: valid stream completed")
invalid_stream.awaitTermination()
print(f"  - {table_name}: quarantine stream completed")

# print(f"\n{table_name} silver layer completed.")
# print(f"  Silver rows:     {spark.sql(f'SELECT COUNT(*) FROM {silver_table}').collect()[0][0]}")
# print(f"  Quarantine rows: {spark.sql(f'SELECT COUNT(*) FROM {quarantine_table}').collect()[0][0]}")

In [0]:
from pyspark.sql.functions import col, when

table_name = "nation"
checkpoint_path = f"/Volumes/{catalog}/silver/checkpoints/{table_name}"
silver_table = f"{catalog}.silver.{table_name}"
quarantine_table = f"{catalog}.silver.{table_name}_quarantine"

# 1. Create silver table if not exists (original columns only, no meta)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {silver_table}
    USING DELTA
    AS SELECT n_nationkey, n_name, n_regionkey, n_comment
    FROM {catalog}.bronze.{table_name} WHERE 1=0
""")

# 2. Create quarantine table if not exists (original columns + _rejection_reason)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {quarantine_table}
    USING DELTA
    AS SELECT n_nationkey, n_name, n_regionkey, n_comment,
              CAST(NULL AS STRING) AS _rejection_reason
    FROM {catalog}.bronze.{table_name} WHERE 1=0
""")

# 3. Read stream from bronze with CDF
source_df = (spark.readStream
    .format("delta")
    .option("readChangeFeed", "true")
    .table(f"{catalog}.bronze.{table_name}"))

# Columns to keep (exclude CDF + meta columns)
exclude_cols = {"_change_type", "_commit_version", "_commit_timestamp",
                "_data_source", "_ingested_at", "_job_run_id"}
silver_cols = [c for c in source_df.columns if c not in exclude_cols]

# 4. Referential integrity: join with silver.region to check n_regionkey exists
region_df = spark.table(f"{catalog}.silver.region").select(
    col("r_regionkey").alias("_ref_regionkey")
)

source_with_ref = source_df.join(
    region_df,
    source_df.n_regionkey == region_df._ref_regionkey,
    "left"
)

# 5. Apply validation rules
#    Rule 1: Primary key (n_nationkey) IS NOT NULL
#    Rule 2: Foreign key (n_regionkey) exists in silver.region
#    Rule 3: Name (n_name) IS NOT NULL AND not empty
#    Only process inserts and update_postimages

valid_df = source_with_ref.filter(
    col("_change_type").isin("insert", "update_postimage") &
    col("n_nationkey").isNotNull() &
    col("n_name").isNotNull() &
    (col("n_name") != "") &
    col("_ref_regionkey").isNotNull()
).select(*silver_cols)

invalid_df = source_with_ref.filter(
    col("_change_type").isin("insert", "update_postimage") &
    (
        col("n_nationkey").isNull() |
        col("n_name").isNull() |
        (col("n_name") == "") |
        col("_ref_regionkey").isNull()
    )
).select(*silver_cols).withColumn("_rejection_reason",
    when(col("n_nationkey").isNull(), "Primary key (n_nationkey) is NULL")
    .when(col("n_name").isNull(), "Name (n_name) is NULL")
    .when(col("n_name") == "", "Name (n_name) is empty")
    .otherwise("Foreign key (n_regionkey) not found in silver.region")
)

# 6. Write valid rows to silver via MERGE (SCD Type 1)
def merge_nation(batch_df, batch_id):
    if batch_df.count() > 0:
        batch_df.createOrReplaceTempView(f"nation_updates_{batch_id}")
        spark.sql(f"""
            MERGE INTO {silver_table} t
            USING nation_updates_{batch_id} s
            ON t.n_nationkey = s.n_nationkey
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)

valid_stream = (valid_df.writeStream
    .foreachBatch(merge_nation)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start())

# 7. Write invalid rows to quarantine
quarantine_checkpoint = f"{checkpoint_path}_quarantine"
dbutils.fs.mkdirs(quarantine_checkpoint)

def quarantine_nation(batch_df, batch_id):
    if batch_df.count() > 0:
        batch_df.write.format("delta").mode("append").saveAsTable(quarantine_table)

invalid_stream = (invalid_df.writeStream
    .foreachBatch(quarantine_nation)
    .option("checkpointLocation", quarantine_checkpoint)
    .trigger(availableNow=True)
    .start())

# 8. Wait for both streams to complete
valid_stream.awaitTermination()
print(f"  - {table_name}: valid stream completed")
invalid_stream.awaitTermination()
print(f"  - {table_name}: quarantine stream completed")

# print(f"\n{table_name} silver layer completed.")
# print(f"  Silver rows:     {spark.sql(f'SELECT COUNT(*) FROM {silver_table}').collect()[0][0]}")
# print(f"  Quarantine rows: {spark.sql(f'SELECT COUNT(*) FROM {quarantine_table}').collect()[0][0]}")

In [0]:
from pyspark.sql.functions import col, when

table_name = "customer"
checkpoint_path = f"/Volumes/{catalog}/silver/checkpoints/{table_name}"
silver_table = f"{catalog}.silver.{table_name}"
quarantine_table = f"{catalog}.silver.{table_name}_quarantine"

# 1. Create silver table if not exists (original columns only, no meta)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {silver_table}
    USING DELTA
    AS SELECT c_custkey, c_name, c_address, c_nationkey, c_phone, c_acctbal, c_mktsegment, c_comment
    FROM {catalog}.bronze.{table_name} WHERE 1=0
""")

# 2. Create quarantine table if not exists (original columns + _rejection_reason)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {quarantine_table}
    USING DELTA
    AS SELECT c_custkey, c_name, c_address, c_nationkey, c_phone, c_acctbal, c_mktsegment, c_comment,
              CAST(NULL AS STRING) AS _rejection_reason
    FROM {catalog}.bronze.{table_name} WHERE 1=0
""")

# 3. Read stream from bronze with CDF
source_df = (spark.readStream
    .format("delta")
    .option("readChangeFeed", "true")
    .table(f"{catalog}.bronze.{table_name}"))

# Columns to keep (exclude CDF + meta columns)
exclude_cols = {"_change_type", "_commit_version", "_commit_timestamp",
                "_data_source", "_ingested_at", "_job_run_id"}
silver_cols = [c for c in source_df.columns if c not in exclude_cols]

# 4. Referential integrity: join with silver.nation to check c_nationkey exists
nation_df = spark.table(f"{catalog}.silver.nation").select(
    col("n_nationkey").alias("_ref_nationkey")
)

source_with_ref = source_df.join(
    nation_df,
    source_df.c_nationkey == nation_df._ref_nationkey,
    "left"
)

# 5. Apply validation rules
#    Rule 1: Primary key (c_custkey) IS NOT NULL
#    Rule 2: Name (c_name) IS NOT NULL AND not empty
#    Rule 3: Foreign key (c_nationkey) exists in silver.nation
#    Only process inserts and update_postimages

valid_df = source_with_ref.filter(
    col("_change_type").isin("insert", "update_postimage") &
    col("c_custkey").isNotNull() &
    col("c_name").isNotNull() &
    (col("c_name") != "") &
    col("_ref_nationkey").isNotNull()
).select(*silver_cols, "_commit_version")

invalid_df = source_with_ref.filter(
    col("_change_type").isin("insert", "update_postimage") &
    (
        col("c_custkey").isNull() |
        col("c_name").isNull() |
        (col("c_name") == "") |
        col("_ref_nationkey").isNull()
    )
).select(*silver_cols).withColumn("_rejection_reason",
    when(col("c_custkey").isNull(), "Primary key (c_custkey) is NULL")
    .when(col("c_name").isNull(), "Name (c_name) is NULL")
    .when(col("c_name") == "", "Name (c_name) is empty")
    .otherwise("Foreign key (c_nationkey) not found in silver.nation")
)

# 6. Write valid rows to silver via MERGE (SCD Type 1)
def merge_customer(batch_df, batch_id):
    if batch_df.count() > 0:
        batch_df.createOrReplaceTempView(f"customer_updates_{batch_id}")
        cols_str = ", ".join(silver_cols)
        spark.sql(f"""
            WITH deduped AS (
                SELECT *,
                    ROW_NUMBER() OVER (PARTITION BY c_custkey ORDER BY _commit_version DESC) AS _rn
                FROM customer_updates_{batch_id}
            )
            MERGE INTO {silver_table} t
            USING (SELECT {cols_str} FROM deduped WHERE _rn = 1) s
            ON t.c_custkey = s.c_custkey
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)

valid_stream = (valid_df.writeStream
    .foreachBatch(merge_customer)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start())

# 7. Write invalid rows to quarantine
quarantine_checkpoint = f"{checkpoint_path}_quarantine"
dbutils.fs.mkdirs(quarantine_checkpoint)

def quarantine_customer(batch_df, batch_id):
    if batch_df.count() > 0:
        batch_df.write.format("delta").mode("append").saveAsTable(quarantine_table)

invalid_stream = (invalid_df.writeStream
    .foreachBatch(quarantine_customer)
    .option("checkpointLocation", quarantine_checkpoint)
    .trigger(availableNow=True)
    .start())

# 8. Wait for both streams to complete
valid_stream.awaitTermination()
print(f"  - {table_name}: valid stream completed")
invalid_stream.awaitTermination()
print(f"  - {table_name}: quarantine stream completed")

In [0]:
from pyspark.sql.functions import col, when

table_name = "supplier"
checkpoint_path = f"/Volumes/{catalog}/silver/checkpoints/{table_name}"
silver_table = f"{catalog}.silver.{table_name}"
quarantine_table = f"{catalog}.silver.{table_name}_quarantine"

# 1. Create silver table if not exists (original columns only, no meta)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {silver_table}
    USING DELTA
    AS SELECT s_suppkey, s_name, s_address, s_nationkey, s_phone, s_acctbal, s_comment
    FROM {catalog}.bronze.{table_name} WHERE 1=0
""")

# 2. Create quarantine table if not exists (original columns + _rejection_reason)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {quarantine_table}
    USING DELTA
    AS SELECT s_suppkey, s_name, s_address, s_nationkey, s_phone, s_acctbal, s_comment,
              CAST(NULL AS STRING) AS _rejection_reason
    FROM {catalog}.bronze.{table_name} WHERE 1=0
""")

# 3. Read stream from bronze with CDF
source_df = (spark.readStream
    .format("delta")
    .option("readChangeFeed", "true")
    .table(f"{catalog}.bronze.{table_name}"))

# Columns to keep (exclude CDF + meta columns)
exclude_cols = {"_change_type", "_commit_version", "_commit_timestamp",
                "_data_source", "_ingested_at", "_job_run_id"}
silver_cols = [c for c in source_df.columns if c not in exclude_cols]

# 4. Referential integrity: join with silver.nation to check s_nationkey exists
nation_df = spark.table(f"{catalog}.silver.nation").select(
    col("n_nationkey").alias("_ref_nationkey")
)

source_with_ref = source_df.join(
    nation_df,
    source_df.s_nationkey == nation_df._ref_nationkey,
    "left"
)

# 5. Apply validation rules
#    Rule 1: Primary key (s_suppkey) IS NOT NULL
#    Rule 2: Name (s_name) IS NOT NULL AND not empty
#    Rule 3: Foreign key (s_nationkey) exists in silver.nation
#    Only process inserts and update_postimages

valid_df = source_with_ref.filter(
    col("_change_type").isin("insert", "update_postimage") &
    col("s_suppkey").isNotNull() &
    col("s_name").isNotNull() &
    (col("s_name") != "") &
    col("_ref_nationkey").isNotNull()
).select(*silver_cols)

invalid_df = source_with_ref.filter(
    col("_change_type").isin("insert", "update_postimage") &
    (
        col("s_suppkey").isNull() |
        col("s_name").isNull() |
        (col("s_name") == "") |
        col("_ref_nationkey").isNull()
    )
).select(*silver_cols).withColumn("_rejection_reason",
    when(col("s_suppkey").isNull(), "Primary key (s_suppkey) is NULL")
    .when(col("s_name").isNull(), "Name (s_name) is NULL")
    .when(col("s_name") == "", "Name (s_name) is empty")
    .otherwise("Foreign key (s_nationkey) not found in silver.nation")
)

# 6. Write valid rows to silver via MERGE (SCD Type 1)
def merge_supplier(batch_df, batch_id):
    if batch_df.count() > 0:
        batch_df.createOrReplaceTempView(f"supplier_updates_{batch_id}")
        spark.sql(f"""
            MERGE INTO {silver_table} t
            USING supplier_updates_{batch_id} s
            ON t.s_suppkey = s.s_suppkey
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)

valid_stream = (valid_df.writeStream
    .foreachBatch(merge_supplier)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start())

# 7. Write invalid rows to quarantine
quarantine_checkpoint = f"{checkpoint_path}_quarantine"
dbutils.fs.mkdirs(quarantine_checkpoint)

def quarantine_supplier(batch_df, batch_id):
    if batch_df.count() > 0:
        batch_df.write.format("delta").mode("append").saveAsTable(quarantine_table)

invalid_stream = (invalid_df.writeStream
    .foreachBatch(quarantine_supplier)
    .option("checkpointLocation", quarantine_checkpoint)
    .trigger(availableNow=True)
    .start())

# 8. Wait for both streams to complete
valid_stream.awaitTermination()
print(f"  - {table_name}: valid stream completed")
invalid_stream.awaitTermination()
print(f"  - {table_name}: quarantine stream completed")

In [0]:
from pyspark.sql.functions import col, when, count as count_fn
from pyspark.sql.window import Window

table_name = "part"
checkpoint_path = f"/Volumes/{catalog}/silver/checkpoints/{table_name}"
silver_table = f"{catalog}.silver.{table_name}"
quarantine_table = f"{catalog}.silver.{table_name}_quarantine"

# 1. Create silver table if not exists (original columns only, no meta)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {silver_table}
    USING DELTA
    AS SELECT p_partkey, p_name, p_mfgr, p_brand, p_type, p_size, p_container, p_retailprice, p_comment
    FROM {catalog}.bronze.{table_name} WHERE 1=0
""")

# 2. Create quarantine table if not exists (original columns + _rejection_reason)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {quarantine_table}
    USING DELTA
    AS SELECT p_partkey, p_name, p_mfgr, p_brand, p_type, p_size, p_container, p_retailprice, p_comment,
              CAST(NULL AS STRING) AS _rejection_reason
    FROM {catalog}.bronze.{table_name} WHERE 1=0
""")

# 3. Read stream from bronze with CDF
source_df = (spark.readStream
    .format("delta")
    .option("readChangeFeed", "true")
    .table(f"{catalog}.bronze.{table_name}"))

# Columns to keep (exclude CDF + meta columns)
exclude_cols = {"_change_type", "_commit_version", "_commit_timestamp",
                "_data_source", "_ingested_at", "_job_run_id"}
silver_cols = [c for c in source_df.columns if c not in exclude_cols]

# 4. Filter to inserts and update_postimages only, select silver columns
filtered_df = source_df.filter(
    col("_change_type").isin("insert", "update_postimage")
).select(*silver_cols)

# 5. Process in foreachBatch: validate, split, merge valid, quarantine invalid
#    Duplicate check requires window functions — not supported on streaming DataFrames,
#    so all validation happens inside foreachBatch where batch_df is a static DataFrame.

def process_part(batch_df, batch_id):
    if batch_df.count() == 0:
        return

    # Detect duplicates within the batch on p_partkey
    w = Window.partitionBy("p_partkey")
    batch_with_dup = batch_df.withColumn("_dup_count", count_fn("*").over(w))

    # Valid: PK not null, not duplicate, size > 0, retailprice > 0, name not null/empty
    valid = batch_with_dup.filter(
        col("p_partkey").isNotNull() &
        (col("_dup_count") == 1) &
        col("p_size").isNotNull() & (col("p_size") > 0) &
        col("p_retailprice").isNotNull() & (col("p_retailprice") > 0) &
        col("p_name").isNotNull() & (col("p_name") != "")
    ).drop("_dup_count")

    # Invalid: any validation fails
    invalid = batch_with_dup.filter(
        col("p_partkey").isNull() |
        (col("_dup_count") > 1) |
        col("p_size").isNull() | (col("p_size") <= 0) |
        col("p_retailprice").isNull() | (col("p_retailprice") <= 0) |
        col("p_name").isNull() |
        (col("p_name") == "")
    ).withColumn("_rejection_reason",
        when(col("p_partkey").isNull(), "Primary key (p_partkey) is NULL")
        .when(col("_dup_count") > 1, "Primary key (p_partkey) is duplicate")
        .when(col("p_size").isNull(), "Size (p_size) is NULL")
        .when(col("p_size") <= 0, "Size (p_size) is not positive")
        .when(col("p_retailprice").isNull(), "Retailprice (p_retailprice) is NULL")
        .when(col("p_retailprice") <= 0, "Retailprice (p_retailprice) is not positive")
        .when(col("p_name").isNull(), "Name (p_name) is NULL")
        .otherwise("Name (p_name) is empty")
    ).drop("_dup_count")

    # MERGE valid rows into silver (SCD Type 1)
    if valid.count() > 0:
        valid.createOrReplaceTempView(f"part_updates_{batch_id}")
        spark.sql(f"""
            MERGE INTO {silver_table} t
            USING part_updates_{batch_id} s
            ON t.p_partkey = s.p_partkey
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)

    # Append invalid rows to quarantine
    if invalid.count() > 0:
        invalid.write.format("delta").mode("append").saveAsTable(quarantine_table)

# 6. Write stream with foreachBatch (single stream handles both valid + quarantine)
stream = (filtered_df.writeStream
    .foreachBatch(process_part)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start())

# 7. Wait for stream to complete
stream.awaitTermination()
print(f"  - {table_name}: stream completed")

In [0]:
from pyspark.sql.functions import col, when, count as count_fn, row_number
from pyspark.sql.window import Window

table_name = "orders"
checkpoint_path = f"/Volumes/{catalog}/silver/checkpoints/{table_name}"
silver_table = f"{catalog}.silver.{table_name}"
quarantine_table = f"{catalog}.silver.{table_name}_quarantine"

# 1. Create silver table if not exists (original columns only, no meta)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {silver_table}
    USING DELTA
    AS SELECT o_orderkey, o_custkey, o_orderstatus, o_totalprice, o_orderdate,
              o_orderpriority, o_clerk, o_shippriority, o_comment
    FROM {catalog}.bronze.{table_name} WHERE 1=0
""")

# 2. Create quarantine table if not exists (original columns + _rejection_reason)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {quarantine_table}
    USING DELTA
    AS SELECT o_orderkey, o_custkey, o_orderstatus, o_totalprice, o_orderdate,
              o_orderpriority, o_clerk, o_shippriority, o_comment,
              CAST(NULL AS STRING) AS _rejection_reason
    FROM {catalog}.bronze.{table_name} WHERE 1=0
""")

# 3. Read stream from bronze with CDF
source_df = (spark.readStream
    .format("delta")
    .option("readChangeFeed", "true")
    .table(f"{catalog}.bronze.{table_name}"))

# Columns to keep (exclude CDF + meta columns)
exclude_cols = {"_change_type", "_commit_version", "_commit_timestamp",
                "_data_source", "_ingested_at", "_job_run_id"}
silver_cols = [c for c in source_df.columns if c not in exclude_cols]

# 4. Filter to inserts and update_postimages only (keep CDF cols for latest-row logic)
filtered_df = source_df.filter(
    col("_change_type").isin("insert", "update_postimage")
)

# 5. Process in foreachBatch: validate, split, merge valid, quarantine invalid
#    Duplicate check (take latest by _commit_version) requires window functions,
#    so all validation happens inside foreachBatch.

def process_orders(batch_df, batch_id):
    if batch_df.count() == 0:
        return

    # Detect duplicates on o_orderkey — keep the latest by _commit_version
    w = Window.partitionBy("o_orderkey").orderBy(col("_commit_version").desc())
    batch_with_dup = batch_df.withColumn("_row_num", row_number().over(w))

    # Referential integrity: join with silver.customer to check o_custkey exists
    customer_df = spark.table(f"{catalog}.silver.customer").select(
        col("c_custkey").alias("_ref_custkey")
    )
    batch_with_ref = batch_with_dup.join(
        customer_df,
        batch_with_dup.o_custkey == customer_df._ref_custkey,
        "left"
    )

    # Valid: PK not null, is latest (row_num == 1), FK exists, price > 0, orderdate in range
    valid = batch_with_ref.filter(
        col("o_orderkey").isNotNull() &
        (col("_row_num") == 1) &
        col("_ref_custkey").isNotNull() &
        col("o_totalprice").isNotNull() & (col("o_totalprice") > 0) &
        col("o_orderdate").isNotNull() &
        (col("o_orderdate") >= "1992-01-01") &
        (col("o_orderdate") <= "1998-12-31")
    ).select(*silver_cols)

    # Invalid: any validation fails
    invalid = batch_with_ref.filter(
        col("o_orderkey").isNull() |
        (col("_row_num") > 1) |
        col("_ref_custkey").isNull() |
        col("o_totalprice").isNull() | (col("o_totalprice") <= 0) |
        col("o_orderdate").isNull() |
        (col("o_orderdate") < "1992-01-01") |
        (col("o_orderdate") > "1998-12-31")
    ).withColumn("_rejection_reason",
        when(col("o_orderkey").isNull(), "Primary key (o_orderkey) is NULL")
        .when(col("_row_num") > 1, "Primary key (o_orderkey) is duplicate (older record)")
        .when(col("_ref_custkey").isNull(), "Foreign key (o_custkey) not found in silver.customer")
        .when(col("o_totalprice").isNull(), "Totalprice (o_totalprice) is NULL")
        .when(col("o_totalprice") <= 0, "Totalprice (o_totalprice) is not positive")
        .when(col("o_orderdate").isNull(), "Order date (o_orderdate) is NULL")
        .when(col("o_orderdate") < "1992-01-01", "Order date (o_orderdate) is before 1992-01-01")
        .otherwise("Order date (o_orderdate) is after 1998-12-31")
    ).select(*silver_cols, "_rejection_reason")

    # MERGE valid rows into silver (SCD Type 1)
    if valid.count() > 0:
        valid.createOrReplaceTempView(f"orders_updates_{batch_id}")
        spark.sql(f"""
            MERGE INTO {silver_table} t
            USING orders_updates_{batch_id} s
            ON t.o_orderkey = s.o_orderkey
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)

    # Append invalid rows to quarantine
    if invalid.count() > 0:
        invalid.write.format("delta").mode("append").saveAsTable(quarantine_table)

# 6. Write stream with foreachBatch (single stream handles both valid + quarantine)
stream = (filtered_df.writeStream
    .foreachBatch(process_orders)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start())

# 7. Wait for stream to complete
stream.awaitTermination()
print(f"  - {table_name}: stream completed")

In [0]:
from pyspark.sql.functions import col, when

table_name = "partsupp"
checkpoint_path = f"/Volumes/{catalog}/silver/checkpoints/{table_name}"
silver_table = f"{catalog}.silver.{table_name}"
quarantine_table = f"{catalog}.silver.{table_name}_quarantine"

# 1. Create silver table if not exists (original columns only, no meta)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {silver_table}
    USING DELTA
    AS SELECT ps_partkey, ps_suppkey, ps_availqty, ps_supplycost, ps_comment
    FROM {catalog}.bronze.{table_name} WHERE 1=0
""")

# 2. Create quarantine table if not exists (original columns + _rejection_reason)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {quarantine_table}
    USING DELTA
    AS SELECT ps_partkey, ps_suppkey, ps_availqty, ps_supplycost, ps_comment,
              CAST(NULL AS STRING) AS _rejection_reason
    FROM {catalog}.bronze.{table_name} WHERE 1=0
""")

# 3. Read stream from bronze with CDF
source_df = (spark.readStream
    .format("delta")
    .option("readChangeFeed", "true")
    .table(f"{catalog}.bronze.{table_name}"))

# Columns to keep (exclude CDF + meta columns)
exclude_cols = {"_change_type", "_commit_version", "_commit_timestamp",
                "_data_source", "_ingested_at", "_job_run_id"}
silver_cols = [c for c in source_df.columns if c not in exclude_cols]

# 4. Referential integrity: join with silver.part and silver.supplier
part_df = spark.table(f"{catalog}.silver.part").select(
    col("p_partkey").alias("_ref_partkey")
)
supplier_df = spark.table(f"{catalog}.silver.supplier").select(
    col("s_suppkey").alias("_ref_suppkey")
)

source_with_ref = source_df.join(
    part_df,
    source_df.ps_partkey == part_df._ref_partkey,
    "left"
).join(
    supplier_df,
    source_df.ps_suppkey == supplier_df._ref_suppkey,
    "left"
)

# 5. Apply validation rules
#    Rule 1: Foreign key (ps_partkey) exists in silver.part
#    Rule 2: Foreign key (ps_suppkey) exists in silver.supplier
#    Rule 3: Quantity (ps_availqty) IS positive (> 0)
#    Rule 4: Supply cost (ps_supplycost) IS positive (> 0)
#    Only process inserts and update_postimages

valid_df = source_with_ref.filter(
    col("_change_type").isin("insert", "update_postimage") &
    col("_ref_partkey").isNotNull() &
    col("_ref_suppkey").isNotNull() &
    col("ps_availqty").isNotNull() & (col("ps_availqty") > 0) &
    col("ps_supplycost").isNotNull() & (col("ps_supplycost") > 0)
).select(*silver_cols)

invalid_df = source_with_ref.filter(
    col("_change_type").isin("insert", "update_postimage") &
    (
        col("_ref_partkey").isNull() |
        col("_ref_suppkey").isNull() |
        col("ps_availqty").isNull() | (col("ps_availqty") <= 0) |
        col("ps_supplycost").isNull() | (col("ps_supplycost") <= 0)
    )
).withColumn("_rejection_reason",
    when(col("_ref_partkey").isNull(), "Foreign key (ps_partkey) not found in silver.part")
    .when(col("_ref_suppkey").isNull(), "Foreign key (ps_suppkey) not found in silver.supplier")
    .when(col("ps_availqty").isNull(), "Quantity (ps_availqty) is NULL")
    .when(col("ps_availqty") <= 0, "Quantity (ps_availqty) is not positive")
    .when(col("ps_supplycost").isNull(), "Supply cost (ps_supplycost) is NULL")
    .otherwise("Supply cost (ps_supplycost) is not positive")
).select(*silver_cols, "_rejection_reason")

# 6. Write valid rows to silver via MERGE (SCD Type 1, composite key)
def merge_partsupp(batch_df, batch_id):
    if batch_df.count() > 0:
        batch_df.createOrReplaceTempView(f"partsupp_updates_{batch_id}")
        spark.sql(f"""
            MERGE INTO {silver_table} t
            USING partsupp_updates_{batch_id} s
            ON t.ps_partkey = s.ps_partkey AND t.ps_suppkey = s.ps_suppkey
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)

valid_stream = (valid_df.writeStream
    .foreachBatch(merge_partsupp)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start())

# 7. Write invalid rows to quarantine
quarantine_checkpoint = f"{checkpoint_path}_quarantine"
dbutils.fs.mkdirs(quarantine_checkpoint)

def quarantine_partsupp(batch_df, batch_id):
    if batch_df.count() > 0:
        batch_df.write.format("delta").mode("append").saveAsTable(quarantine_table)

invalid_stream = (invalid_df.writeStream
    .foreachBatch(quarantine_partsupp)
    .option("checkpointLocation", quarantine_checkpoint)
    .trigger(availableNow=True)
    .start())

# 8. Wait for both streams to complete
valid_stream.awaitTermination()
print(f"  - {table_name}: valid stream completed")
invalid_stream.awaitTermination()
print(f"  - {table_name}: quarantine stream completed")

In [0]:
from pyspark.sql.functions import col, when

table_name = "lineitem"
checkpoint_path = f"/Volumes/{catalog}/silver/checkpoints/{table_name}"
silver_table = f"{catalog}.silver.{table_name}"
quarantine_table = f"{catalog}.silver.{table_name}_quarantine"

# 1. Create silver table if not exists (original columns only, no meta)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {silver_table}
    USING DELTA
    AS SELECT l_orderkey, l_partkey, l_suppkey, l_linenumber, l_quantity, l_extendedprice,
              l_discount, l_tax, l_returnflag, l_linestatus, l_shipdate, l_commitdate,
              l_receiptdate, l_shipinstruct, l_shipmode, l_comment
    FROM {catalog}.bronze.{table_name} WHERE 1=0
""")

# 2. Create quarantine table if not exists (original columns + _rejection_reason)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {quarantine_table}
    USING DELTA
    AS SELECT l_orderkey, l_partkey, l_suppkey, l_linenumber, l_quantity, l_extendedprice,
              l_discount, l_tax, l_returnflag, l_linestatus, l_shipdate, l_commitdate,
              l_receiptdate, l_shipinstruct, l_shipmode, l_comment,
              CAST(NULL AS STRING) AS _rejection_reason
    FROM {catalog}.bronze.{table_name} WHERE 1=0
""")

# 3. Read stream from bronze with CDF
source_df = (spark.readStream
    .format("delta")
    .option("readChangeFeed", "true")
    .table(f"{catalog}.bronze.{table_name}"))

# Columns to keep (exclude CDF + meta columns)
exclude_cols = {"_change_type", "_commit_version", "_commit_timestamp",
                "_data_source", "_ingested_at", "_job_run_id"}
silver_cols = [c for c in source_df.columns if c not in exclude_cols]

# 4. Referential integrity: join with silver.orders, silver.part, silver.supplier
orders_df = spark.table(f"{catalog}.silver.orders").select(
    col("o_orderkey").alias("_ref_orderkey")
)
part_df = spark.table(f"{catalog}.silver.part").select(
    col("p_partkey").alias("_ref_partkey")
)
supplier_df = spark.table(f"{catalog}.silver.supplier").select(
    col("s_suppkey").alias("_ref_suppkey")
)

source_with_ref = source_df.join(
    orders_df, source_df.l_orderkey == orders_df._ref_orderkey, "left"
).join(
    part_df, source_df.l_partkey == part_df._ref_partkey, "left"
).join(
    supplier_df, source_df.l_suppkey == supplier_df._ref_suppkey, "left"
)

# 5. Apply validation rules
#    Rule 1: FK (l_orderkey) exists in silver.orders
#    Rule 2: FK (l_partkey) exists in silver.part
#    Rule 3: FK (l_suppkey) exists in silver.supplier
#    Rule 4: l_linenumber IS NOT NULL
#    Rule 5: l_quantity > 0, l_extendedprice > 0
#    Rule 6: l_discount between 0 and 0.10 (0-10%)
#    Rule 7: l_tax between 0 and 0.08 (0-8%)
#    Rule 8: l_shipdate, l_commitdate, l_receiptdate between 1992-01-01 and 1998-12-31
#    Only process inserts and update_postimages

valid_df = source_with_ref.filter(
    col("_change_type").isin("insert", "update_postimage") &
    col("_ref_orderkey").isNotNull() &
    col("_ref_partkey").isNotNull() &
    col("_ref_suppkey").isNotNull() &
    col("l_linenumber").isNotNull() &
    col("l_quantity").isNotNull() & (col("l_quantity") > 0) &
    col("l_extendedprice").isNotNull() & (col("l_extendedprice") > 0) &
    col("l_discount").isNotNull() & (col("l_discount") >= 0) & (col("l_discount") <= 0.10) &
    col("l_tax").isNotNull() & (col("l_tax") >= 0) & (col("l_tax") <= 0.08) &
    col("l_shipdate").isNotNull() & (col("l_shipdate") >= "1992-01-01") & (col("l_shipdate") <= "1998-12-31") &
    col("l_commitdate").isNotNull() & (col("l_commitdate") >= "1992-01-01") & (col("l_commitdate") <= "1998-12-31") &
    col("l_receiptdate").isNotNull() & (col("l_receiptdate") >= "1992-01-01") & (col("l_receiptdate") <= "1998-12-31")
).select(*silver_cols)

invalid_df = source_with_ref.filter(
    col("_change_type").isin("insert", "update_postimage") &
    (
        col("_ref_orderkey").isNull() |
        col("_ref_partkey").isNull() |
        col("_ref_suppkey").isNull() |
        col("l_linenumber").isNull() |
        col("l_quantity").isNull() | (col("l_quantity") <= 0) |
        col("l_extendedprice").isNull() | (col("l_extendedprice") <= 0) |
        col("l_discount").isNull() | (col("l_discount") < 0) | (col("l_discount") > 0.10) |
        col("l_tax").isNull() | (col("l_tax") < 0) | (col("l_tax") > 0.08) |
        col("l_shipdate").isNull() | (col("l_shipdate") < "1992-01-01") | (col("l_shipdate") > "1998-12-31") |
        col("l_commitdate").isNull() | (col("l_commitdate") < "1992-01-01") | (col("l_commitdate") > "1998-12-31") |
        col("l_receiptdate").isNull() | (col("l_receiptdate") < "1992-01-01") | (col("l_receiptdate") > "1998-12-31")
    )
).withColumn("_rejection_reason",
    when(col("_ref_orderkey").isNull(), "Foreign key (l_orderkey) not found in silver.orders")
    .when(col("_ref_partkey").isNull(), "Foreign key (l_partkey) not found in silver.part")
    .when(col("_ref_suppkey").isNull(), "Foreign key (l_suppkey) not found in silver.supplier")
    .when(col("l_linenumber").isNull(), "Line number (l_linenumber) is NULL")
    .when(col("l_quantity").isNull(), "Quantity (l_quantity) is NULL")
    .when(col("l_quantity") <= 0, "Quantity (l_quantity) is not positive")
    .when(col("l_extendedprice").isNull(), "Extended price (l_extendedprice) is NULL")
    .when(col("l_extendedprice") <= 0, "Extended price (l_extendedprice) is not positive")
    .when(col("l_discount").isNull(), "Discount (l_discount) is NULL")
    .when((col("l_discount") < 0) | (col("l_discount") > 0.10), "Discount (l_discount) is not between 0-10%")
    .when(col("l_tax").isNull(), "Tax (l_tax) is NULL")
    .when((col("l_tax") < 0) | (col("l_tax") > 0.08), "Tax (l_tax) is not between 0-8%")
    .when(col("l_shipdate").isNull(), "Ship date (l_shipdate) is NULL")
    .when((col("l_shipdate") < "1992-01-01") | (col("l_shipdate") > "1998-12-31"), "Ship date (l_shipdate) not in 1992-01-01 to 1998-12-31")
    .when(col("l_commitdate").isNull(), "Commit date (l_commitdate) is NULL")
    .when((col("l_commitdate") < "1992-01-01") | (col("l_commitdate") > "1998-12-31"), "Commit date (l_commitdate) not in 1992-01-01 to 1998-12-31")
    .when(col("l_receiptdate").isNull(), "Receipt date (l_receiptdate) is NULL")
    .otherwise("Receipt date (l_receiptdate) not in 1992-01-01 to 1998-12-31")
).select(*silver_cols, "_rejection_reason")

# 6. Write valid rows to silver via MERGE (SCD Type 1, composite key)
def merge_lineitem(batch_df, batch_id):
    if batch_df.count() > 0:
        batch_df.createOrReplaceTempView(f"lineitem_updates_{batch_id}")
        spark.sql(f"""
            MERGE INTO {silver_table} t
            USING lineitem_updates_{batch_id} s
            ON t.l_orderkey = s.l_orderkey AND t.l_linenumber = s.l_linenumber
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)

valid_stream = (valid_df.writeStream
    .foreachBatch(merge_lineitem)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start())

# 7. Write invalid rows to quarantine
quarantine_checkpoint = f"{checkpoint_path}_quarantine"
dbutils.fs.mkdirs(quarantine_checkpoint)

def quarantine_lineitem(batch_df, batch_id):
    if batch_df.count() > 0:
        batch_df.write.format("delta").mode("append").saveAsTable(quarantine_table)

invalid_stream = (invalid_df.writeStream
    .foreachBatch(quarantine_lineitem)
    .option("checkpointLocation", quarantine_checkpoint)
    .trigger(availableNow=True)
    .start())

# 8. Wait for both streams to complete
valid_stream.awaitTermination()
print(f"  - {table_name}: valid stream completed")
invalid_stream.awaitTermination()
print(f"  - {table_name}: quarantine stream completed")